# CDM customer fact — column exploration (notebook 1)

Source: `clhhd01p_clh.cdm_customer_fact_v` (a view). Goal: understand every column before choosing what to bring next to PKG — credit risk (rating, PD, watch / non-accrual), credit exposure (commitment, outstanding), revenue / profitability, deposits and products.

Cells
1. Setup and config
2. Metadata — column list, types, comments, view definition (source tables), 5 sample rows side by side
3. Grain and time — total rows, date columns, snapshot column, rows per snapshot, key candidates and an exact duplicate check
4. Column profile on the latest snapshot — fill, distinct, top values, numeric spread and sums, grouped by theme; dead columns listed; saved to CSV

Themes are name-based hints (regex on column name and comment), not ground truth. Re-run from cell 3 after changing SNAPSHOT_COL / SNAPSHOT_VALUE / SAMPLE_FRACTION.

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("cdm_customer_fact_explore")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
TABLE            = "clhhd01p_clh.cdm_customer_fact_v"
SNAPSHOT_COL     = "auto"     # "auto" = detect; None = treat as current-state (no snapshot filter); or a column name
SNAPSHOT_VALUE   = "latest"   # "latest" or "YYYY-MM-DD"
SAMPLE_FRACTION  = None       # e.g. 0.1 to profile a sample of the chosen snapshot; None = all rows
SAMPLE_ROWS      = 5          # rows shown side by side in cell 2
CAT_MAX_DISTINCT = 200        # columns with <= this many distinct values get a top-values list
TOP_K            = 5
BATCH_COLS       = 40         # columns per aggregation pass in cell 4
RSD              = 0.02       # approx_count_distinct relative error
VIEW_TEXT_CHARS  = 4000
OUT_DIR          = "../explore/cdm_customer_fact"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)} · pandas {esc(pd.__version__)} · numpy {esc(np.__version__)}</p>",
      tbl(pd.DataFrame([("TABLE", TABLE), ("SNAPSHOT_COL", SNAPSHOT_COL), ("SNAPSHOT_VALUE", SNAPSHOT_VALUE),
                        ("SAMPLE_FRACTION", SAMPLE_FRACTION), ("CAT_MAX_DISTINCT", CAT_MAX_DISTINCT),
                        ("OUT_DIR", os.path.abspath(OUT_DIR))], columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Metadata: columns, types, comments, view definition, sample rows
# ============================================================
try:
    desc = spark.sql(f"DESCRIBE TABLE EXTENDED {TABLE}").toPandas()
except Exception as e:
    raise RuntimeError(f"Cannot describe {TABLE} from Spark: {str(e)[:400]}")

# DESCRIBE EXTENDED layout: columns, blank row, [# Partition Information ...], # Detailed Table Information
comments, part_cols, info, section = {}, [], {}, "cols"
for name, dtype, cmt in desc[["col_name", "data_type", "comment"]].itertuples(index=False):
    name = (name or "").strip()
    if name.startswith("#"):
        if "Partition" in name: section = "part"
        elif "Detailed" in name: section = "info"
        continue
    if name == "":
        if section == "cols": section = "gap"
        continue
    if section == "cols":
        comments[name] = (cmt or "").strip()
    elif section == "part":
        part_cols.append(name)
    elif section == "info":
        info[name] = (dtype or "").strip()

view_text = info.get("View Text") or info.get("View Original Text") or ""
base_tables = sorted(set(m.lower() for m in re.findall(
    r"\b(?:from|join)\s+`?([A-Za-z_]\w*\.[A-Za-z_]\w*)`?", view_text, flags=re.I)))

# Fail fast if Spark cannot read the view (e.g. Impala-only SQL in the view definition)
try:
    df_all = spark.table(TABLE)
    # cast to text: PySpark 3.3 toPandas fails on timestamp columns under pandas 2
    sample_pdf = df_all.limit(SAMPLE_ROWS).select(
        [F.col(f"`{c}`").cast("string").alias(c) for c in df_all.columns]).toPandas()
except Exception as e:
    show(["<h3 class='warn'>Spark cannot read the view</h3>", f"<pre>{esc(str(e)[:1500])}</pre>",
          "<h4>View text</h4>", f"<pre>{esc(view_text[:VIEW_TEXT_CHARS])}</pre>",
          f"<p>Base tables referenced: {esc(', '.join(base_tables) or '(none parsed)')}</p>"])
    raise RuntimeError("Spark cannot read the view; read its base tables directly instead (see output above).")

schema = [(f.name, f.dataType) for f in df_all.schema.fields]
col_meta = pd.DataFrame({
    "#": range(1, len(schema) + 1),
    "column": [c for c, _ in schema],
    "type": [d.simpleString() for _, d in schema],
    "kind": [kind_of(d) for _, d in schema],
    "comment": [comments.get(c, "") for c, _ in schema],
})

def _cell(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    s = str(v)
    return s if len(s) <= 40 else s[:37] + "..."

side = col_meta[["#", "column", "type", "comment"]].copy()
for i in range(len(sample_pdf)):
    side[f"row{i + 1}"] = [_cell(sample_pdf.iloc[i, j]) for j in range(len(schema))]
if (side["comment"] == "").all():
    side = side.drop(columns=["comment"])

facts = pd.DataFrame([
    ("Type", info.get("Type", "")),
    ("Owner", info.get("Owner", "")),
    ("Created", info.get("Created Time", "")),
    ("Columns", len(schema)),
    ("Partition columns", ", ".join(part_cols) or "(none)"),
    ("Columns with comments", int((col_meta["comment"] != "").sum())),
    ("Base tables in view text", ", ".join(base_tables) or "(none parsed)"),
], columns=["item", "value"])
kinds = col_meta["kind"].value_counts().rename_axis("kind").reset_index(name="columns")

vt = view_text[:VIEW_TEXT_CHARS] + (f"\n... ({len(view_text) - VIEW_TEXT_CHARS} more characters)"
                                    if len(view_text) > VIEW_TEXT_CHARS else "")
show(["<h3>Metadata</h3>", tbl(facts), "<h4>Column kinds</h4>", tbl(kinds),
      "<h4>View definition</h4>", f"<pre>{esc(vt) or '(not a view or no text)'}</pre>",
      f"<h4>Columns with {len(sample_pdf)} sample rows side by side</h4>", tbl(side)])


In [ ]:
# ============================================================
# 3. Grain and time: rows, date columns, snapshot, key candidates
# ============================================================
date_cols = [(c, d) for c, d in schema if is_date_like(c, d)]

# One pass over the whole view: total rows + per date column fill / parse / range / placeholders
aggs = [F.count(F.lit(1)).alias("n")]
for i, (c, d) in enumerate(date_cols):
    p = date_expr(c, d)
    aggs += [F.count(qcol(c)).alias(f"d{i}_nn"), F.count(p).alias(f"d{i}_np"),
             F.approx_count_distinct(p, RSD).alias(f"d{i}_nd"),
             F.min(real_date(p)).alias(f"d{i}_min"), F.max(real_date(p)).alias(f"d{i}_max"),
             F.sum(F.when(p.isNotNull() & ~F.year(p).between(1901, 2099), 1).otherwise(0)).alias(f"d{i}_ph")]
g = df_all.agg(*aggs).collect()[0].asDict()
N_ALL = g["n"]
if N_ALL == 0:
    raise RuntimeError(f"{TABLE} returned 0 rows")

SNAP_NAME = re.compile(r"(as_?of|snap|rpt|report|period|eff|busn|bus_dt|process|month_end|me_dt|cob|load)")
drows = []
for i, (c, d) in enumerate(date_cols):
    nn, npar, nd = g[f"d{i}_nn"], g[f"d{i}_np"], g[f"d{i}_nd"]
    drows.append({"column": c, "type": d.simpleString(), "fill": nn / N_ALL,
                  "parsed of filled": (npar / nn) if nn else None, "distinct dates": nd,
                  "min": g[f"d{i}_min"], "max": g[f"d{i}_max"], "placeholder rows": g[f"d{i}_ph"]})
dstat = pd.DataFrame(drows)

# Snapshot column: snapshot-like name, (almost) always filled and parseable, few distinct values.
# Order: varies across rows (>= 2 dates) first, then business dates before load timestamps, then fewest dates.
snap_reason = ""
if SNAPSHOT_COL == "auto":
    cands = [r for r in drows if SNAP_NAME.search(r["column"].lower()) and r["fill"] >= 0.99
             and (r["parsed of filled"] or 0) >= 0.99 and 1 <= r["distinct dates"] <= 5000]
    cands.sort(key=lambda r: (r["distinct dates"] < 2,
                              r["type"] == "timestamp" or "load" in r["column"].lower(),
                              r["distinct dates"]))
    SNAP = cands[0]["column"] if cands else None
    snap_reason = (f"auto: picked from {len(cands)} candidate(s) ({', '.join(r['column'] for r in cands)})"
                   if cands else "auto: no snapshot-like date column found -> treated as current-state")
else:
    SNAP = SNAPSHOT_COL
    snap_reason = "set in config"

parts = ["<h3>Grain and time</h3>", f"<p>Total rows in view: <b>{N_ALL:,}</b></p>", "<h4>Date-like columns</h4>"]
dshow = dstat.copy()
if len(dshow):
    dshow["fill"] = dshow["fill"].map(fpct)
    dshow["parsed of filled"] = dshow["parsed of filled"].map(fpct)
    dshow["distinct dates"] = dshow["distinct dates"].map(lambda x: f"{x:,}")
    dshow["placeholder rows"] = dshow["placeholder rows"].map(lambda x: f"{x:,}")
parts.append(tbl(dshow))

if SNAP:
    sdt = dict(schema)[SNAP]
    snap_p = date_expr(SNAP, sdt)
    per = (df_all.groupBy(snap_p.alias("snapshot")).count()
                 .orderBy(F.col("snapshot").desc_nulls_last()).limit(24).toPandas())
    snap_value = (per["snapshot"].dropna().max() if SNAPSHOT_VALUE == "latest"
                  else _dt.date.fromisoformat(SNAPSHOT_VALUE))
    df_prof = df_all.filter(snap_p == F.lit(snap_value))
    per["rows"] = per.pop("count").map(lambda x: f"{x:,}")
    parts += [f"<h4>Snapshot column: {esc(SNAP)}</h4><p class='note'>{esc(snap_reason)}. Profiling snapshot "
              f"<b>{esc(snap_value)}</b>.</p>", "<p class='note'>Rows per snapshot (latest 24):</p>", tbl(per)]
else:
    snap_value = None
    df_prof = df_all
    parts += [f"<h4>No snapshot filter</h4><p class='note'>{esc(snap_reason)}</p>"]

if SAMPLE_FRACTION:
    df_prof = df_prof.sample(False, float(SAMPLE_FRACTION), seed=7)

try:
    _prev_prof.unpersist()
except NameError:
    pass
df_prof = df_prof.persist(StorageLevel.MEMORY_AND_DISK)
_prev_prof = df_prof

# Key candidates within the profiled frame (approx distinct), then an exact duplicate check on the best one
KEY_NAME = re.compile(r"(mdm|pwr|power|(^|_)cif|cust|party|obligor|oblg|(^|_)tin(_|$)|duns|(^|_)lei(_|$)|(^|_)ecn|(^|_)ucn|_id$|^id$|_key$|_nbr$|_num$|_no$)")
NOT_KEY = re.compile(r"(_nm$|_name$|name|_desc$|_amt$|_cnt$|_ind$|_flg$|_cd$|_type$|_pct$)")
key_cols = [c for c, d in schema if KEY_NAME.search(c.lower()) and not NOT_KEY.search(c.lower())
            and kind_of(d) in ("str", "num") and c not in {x for x, _ in date_cols}]
kaggs = [F.count(F.lit(1)).alias("n")]
for i, c in enumerate(key_cols):
    kaggs += [F.count(qcol(c)).alias(f"k{i}_nn"), F.approx_count_distinct(qcol(c), 0.01).alias(f"k{i}_nd")]
k = df_prof.agg(*kaggs).collect()[0].asDict()
N_PROF = k["n"]
krows = [{"column": c, "fill": fpct(k[f"k{i}_nn"] / N_PROF if N_PROF else None),
          "distinct (approx)": f"{k[f'k{i}_nd']:,}",
          "rows per value": f"{k[f'k{i}_nn'] / k[f'k{i}_nd']:.2f}" if k[f"k{i}_nd"] else ""}
         for i, c in enumerate(key_cols)]
parts += [f"<h4>Profiled frame</h4><p>Rows: <b>{N_PROF:,}</b>"
          + (f" (sample fraction {SAMPLE_FRACTION})" if SAMPLE_FRACTION else "") + "</p>",
          "<h4>Key candidates (name-based)</h4>", tbl(pd.DataFrame(krows))]

if key_cols and N_PROF:
    best_i = max(range(len(key_cols)), key=lambda i: k[f"k{i}_nd"])
    best = key_cols[best_i]
    dup = (df_prof.filter(qcol(best).isNotNull()).groupBy(qcol(best).alias("v")).count()
                  .filter("count > 1"))
    dsum = dup.agg(F.count(F.lit(1)).alias("vals"), F.sum("count").alias("rows")).collect()[0]
    top_dup = dup.orderBy(F.col("count").desc()).limit(5).toPandas()
    verdict = ("one row per value -> this is the grain" if not dsum["vals"]
               else f"{dsum['vals']:,} values repeat, covering {dsum['rows']:,} rows -> grain is finer than {best}")
    parts += [f"<h4>Exact duplicate check on {esc(best)}</h4><p>{esc(verdict)}</p>"]
    if len(top_dup):
        parts += ["<p class='note'>Most repeated values:</p>", tbl(top_dup.rename(columns={"v": best, "count": "rows"}))]

show(parts)


In [ ]:
# ============================================================
# 4. Column profile (profiled frame from cell 3), grouped by theme
# ============================================================
THEMES = [
    ("meta / load", r"(etl|(^|_)load|insert|updt|(^|_)upd(_|$)|src_sys|source_sys|batch|hdfs|audit|create_ts|created)"),
    ("credit risk", r"((^|_)pd(_|$)|prob\w*def|default|(^|_)dflt|rating|(^|_)rtg|grade|risk|(^|_)rr(_|$)|watch|critic|classif|non_?accr|nonperf|(^|_)npl|delinq|(^|_)dpd|past_?due|chg_?off|charge_?off|(^|_)lgd|impair|cecl|allowance|(^|_)alll|reserve|bankrupt|covenant|fico|score|(^|_)ews|early_warn|distress|restructur|(^|_)tdr|forbear)"),
    ("exposure / credit", r"(expos|(^|_)ead(_|$)|commit|outst|(^|_)osb|limit|(^|_)lmt|loan|(^|_)line|util|collat|(^|_)ltv|undrawn|unfund|drawn|principal|credit|(^|_)crd|(^|_)lc(_|$)|guarant|derivat|swap|lease|mortg|(^|_)cre(_|$))"),
    ("revenue / profit", r"(revenue|(^|_)rev(_|$)|income|(^|_)fee|(^|_)nii|(^|_)nie|interest|spread|margin|profit|raroc|rorac|(^|_)roe|(^|_)roa|(^|_)ftp|wallet|earning|(^|_)pnl|expense|(^|_)eva|contrib|(^|_)ytd|(^|_)ttm)"),
    ("deposits / products / TM", r"(deposit|(^|_)dep(_|$)|(^|_)dda|checking|saving|(^|_)mmda|cert\w*dep|(^|_)cd_(amt|bal)|sweep|(^|_)tm(_|$)|treasury|product|(^|_)prod|(^|_)acct|account|card|merchant|(^|_)ach|wire|lockbox|payabl|receiv|(^|_)bal(_|$)|balance)"),
    ("identity / linkage", r"(mdm|pwr|power|(^|_)cif|cust\w*_id|(^|_)party|(^|_)tin(_|$)|(^|_)ein|duns|(^|_)lei(_|$)|naics|(^|_)sic(_|$)|ultimate|parent|family|hier|obligor|oblg|(^|_)ecn|(^|_)ucn|(^|_)id$|_key$|_nbr$)"),
    ("relationship / segment", r"((^|_)rm(_|$)|officer|banker|(^|_)lob|segment|(^|_)seg(_|$)|region|market|branch|tenure|(^|_)open|since|start|status|(^|_)type|(^|_)name|(^|_)nm$|addr|city|state|(^|_)zip|cntry|country|industry|sector|size|employee|sales|annual)"),
]
THEME_ORDER = [t for t, _ in THEMES[1:]] + ["dates", "other", "meta / load"]
_THEME_RE = [(t, re.compile(p)) for t, p in THEMES]
date_set = {c for c, _ in date_cols}

def themes_of(c):
    name, cmt = c.lower(), comments.get(c, "").lower()   # matched separately so name anchors ($) work
    hits = [t for t, rx in _THEME_RE if rx.search(name) or (cmt and rx.search(cmt))]
    if not hits:
        hits = ["dates"] if c in date_set else ["other"]
    return hits

# ---- pass 1: per-column aggregates, in batches ----
res = {}
for b in range(0, len(schema), BATCH_COLS):
    aggs = []
    for j, (c, d) in enumerate(schema[b:b + BATCH_COLS], start=b):
        kd, col, pre = kind_of(d), qcol(c), f"c{j}_"
        aggs.append(F.count(col).alias(pre + "nn"))
        if kd == "complex":
            continue
        aggs += [F.approx_count_distinct(col, RSD).alias(pre + "nd"),
                 F.first(col.cast("string"), ignorenulls=True).alias(pre + "ex")]
        if kd == "num":
            x = col.cast("double")
            aggs += [F.min(x).alias(pre + "min"), F.max(x).alias(pre + "max"), F.avg(x).alias(pre + "mean"),
                     F.sum(x).alias(pre + "sum"),
                     F.sum(F.when(x == 0, 1).otherwise(0)).alias(pre + "zero"),
                     F.sum(F.when(x < 0, 1).otherwise(0)).alias(pre + "neg"),
                     F.percentile_approx(x, [0.01, 0.5, 0.99], 1000).alias(pre + "pct")]
        elif kd == "str":
            s = F.trim(col)
            aggs += [F.sum(F.when(s == "", 1).otherwise(0)).alias(pre + "blank"),
                     F.max(F.length(col)).alias(pre + "maxlen"),
                     F.sum(F.when(s.cast("double").isNotNull(), 1).otherwise(0)).alias(pre + "numlike")]
        if c in date_set:
            p = date_expr(c, d)
            aggs += [F.min(real_date(p)).alias(pre + "dmin"), F.max(real_date(p)).alias(pre + "dmax"),
                     F.count(p).alias(pre + "dparsed")]
    res.update(df_prof.agg(*aggs).collect()[0].asDict())

# ---- pass 2: top values for low-cardinality columns (one melt + group-by) ----
cat_idx = [j for j, (c, d) in enumerate(schema)
           if kind_of(d) != "complex" and (res.get(f"c{j}_nd") or 0) <= CAT_MAX_DISTINCT]
top_vals = {}
if cat_idx and N_PROF:
    melt = df_prof.select(F.explode(F.array(*[
        F.struct(F.lit(j).alias("k"), qcol(schema[j][0]).cast("string").alias("v")) for j in cat_idx])).alias("kv"))
    tv = melt.groupBy("kv.k", "kv.v").count().toPandas()
    for j, grp in tv.groupby("k"):
        grp = grp.sort_values("count", ascending=False).head(TOP_K)
        top_vals[int(j)] = "; ".join(f"{'(null)' if v is None else (v if len(str(v)) <= 30 else str(v)[:27] + '...')}"
                                     f" {100 * n / N_PROF:.1f}%" for v, n in zip(grp["v"], grp["count"]))

# ---- assemble ----
rows = []
for j, (c, d) in enumerate(schema):
    kd, pre = kind_of(d), f"c{j}_"
    nn, nd = res.get(pre + "nn") or 0, res.get(pre + "nd")
    pct = res.get(pre + "pct") or [None, None, None]
    flags = []
    if kd == "str" and nn:
        if res.get(pre + "blank"): flags.append(f"blank {fpct(res[pre + 'blank'] / nn)}")
        if res.get(pre + "numlike", 0) / nn >= 0.95: flags.append("numeric stored as text")
        flags.append(f"maxlen {res.get(pre + 'maxlen')}")
    if c in date_set and nn:
        flags.append(f"dates {res.get(pre + 'dmin')} .. {res.get(pre + 'dmax')}")
        if res.get(pre + "dparsed", 0) < nn: flags.append(f"unparsed {fpct(1 - res[pre + 'dparsed'] / nn)}")
    th = themes_of(c)
    no_stats = c in date_set or c in key_cols   # sums / percentiles of ids and yyyymmdd numbers are meaningless
    if no_stats:
        pct = [None, None, None]
    rows.append({
        "#": j + 1, "column": c, "type": d.simpleString(), "theme": th[0], "also": ", ".join(th[1:]),
        "fill": nn / N_PROF if N_PROF else None, "distinct": nd,
        "top values / example": top_vals.get(j, "" if res.get(pre + "ex") is None else f"e.g. {str(res[pre + 'ex'])[:40]}"),
        "min": None if no_stats else res.get(pre + "min"), "p01": pct[0], "p50": pct[1], "p99": pct[2],
        "max": None if no_stats else res.get(pre + "max"),
        "mean": None if no_stats else res.get(pre + "mean"), "sum": None if no_stats else res.get(pre + "sum"),
        "zero": (res.get(pre + "zero") / nn) if kd == "num" and nn and not no_stats else None,
        "neg": (res.get(pre + "neg") / nn) if kd == "num" and nn and not no_stats else None,
        "flags": "; ".join(flags), "comment": comments.get(c, ""),
    })
prof = pd.DataFrame(rows)
prof["dead"] = (prof["fill"].fillna(0) == 0) | (prof["distinct"].fillna(0) <= 1)

csv_path = os.path.join(OUT_DIR, "column_profile.csv")
prof.assign(snapshot=str(snap_value) if snap_value else "", rows_profiled=N_PROF).to_csv(csv_path, index=False)

# ---- display ----
def pretty(p):
    q = p.copy()
    q["fill"] = q["fill"].map(fpct)
    q["distinct"] = q["distinct"].map(lambda x: "" if pd.isna(x) else f"{int(x):,}")
    for c in ("min", "p01", "p50", "p99", "max", "mean", "sum"):
        q[c] = q[c].map(fnum)
    for c in ("zero", "neg"):
        q[c] = q[c].map(fpct)
    q = q.drop(columns=["theme", "dead"])
    if (q["comment"] == "").all():
        q = q.drop(columns=["comment"])
    num_cols = ["min", "p01", "p50", "p99", "max", "mean", "sum", "zero", "neg"]
    if (q[num_cols] == "").all().all():
        q = q.drop(columns=num_cols)
    return q

summ = (prof.groupby("theme")
            .agg(columns=("column", "size"), alive=("dead", lambda s: int((~s).sum())),
                 median_fill=("fill", "median"))
            .reindex([t for t in THEME_ORDER if t in set(prof["theme"])]).reset_index())
summ["median_fill"] = summ["median_fill"].map(fpct)
dead = prof.loc[prof["dead"], "column"].tolist()

parts = [f"<h3>Column profile — {N_PROF:,} rows"
         + (f", snapshot {esc(snap_value)}" if snap_value else "") + "</h3>",
         "<p class='note'>Theme = first regex hit on name + comment (hint only). Percentiles approximate. "
         "Amounts in K / M / B / T. Top values show share of all profiled rows.</p>",
         "<h4>Themes</h4>", tbl(summ),
         f"<h4>Empty or constant in this snapshot: {len(dead)}</h4><p>{esc(', '.join(dead)) or '(none)'}</p>"]
for t in THEME_ORDER:
    sub = prof[(prof["theme"] == t) & ~prof["dead"]]
    if len(sub):
        parts += [f"<h4>{esc(t)} — {len(sub)} columns</h4>", tbl(pretty(sub))]
parts.append(f"<p class='note'>Full profile saved: {esc(os.path.abspath(csv_path))}</p>")
show(parts)
